# Annotation overview

Distributions of reviewed letters, reference categories, markers, target works, and temporal perspectives.

In [ ]:
from shared import paths
from IPython.display import display


## Prepare annotation tables

In [ ]:
from annotations.tables import prepare_tables, as_list
from shared.labels import CATEGORY_INDEX
from shared.text import DIVIDER, ABBREVIATIONS, protect_abbreviations, restore_abbreviations
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import re

paths.prepare_directories()
tables = prepare_tables()
letters = tables['letters']
reviewed_letters = tables['reviewed_letters']
letters_df = tables['letters_df']
sentences_df = tables['sentences_df']
traces_df = tables['traces_df']
annotations_df = tables['annotations_df']
sentence_annotation_df = tables['sentence_annotation_df']
all_sentences = tables['all_sentences']
NUM_CATEGORIES = len(CATEGORY_INDEX)
id2label = {i: label for label, i in CATEGORY_INDEX.items()}
skipped_no_relevant_block = tables['skipped_no_relevant_block']


## Overview

In [ ]:
def pct(n, d):
    return f'{n / d:.2%}' if d else 'n/a'


def make_overview_table(rows):
    return pd.DataFrame(rows, columns=['Metric', 'Value'])
total_sentences = len(sentences_df)
num_ref = int(sentences_df['has_reference'].sum())
num_noref = total_sentences - num_ref
multi_label_sentences_df = sentences_df[sentences_df['has_reference'] & (sentences_df['category_vector'].apply(sum) > 1)]
num_multi = len(multi_label_sentences_df)
percent_multi = round(100 * num_multi / num_ref, 2) if num_ref else 0
avg_labels_per_ref = sentences_df.loc[sentences_df['has_reference'], 'category_vector'].apply(sum).mean() if num_ref else 0
overview = make_overview_table(
    [
        ('Letters loaded from pickle', f'{len(letters):,}'),
        ('Reviewed letters', f'{len(reviewed_letters):,}'),
        ('Reviewed letters with extracted sentences', f'{len(letters_df):,}'),
        (
            'Skipped reviewed letters without extracted relevant text block',
            f'{len(skipped_no_relevant_block):,}',
        ),
        ('Letters with intertextual traces', f"{letters_df['has_intertextual_traces'].sum():,}"),
        (
            'Letters without intertextual traces',
            f"{(~letters_df['has_intertextual_traces']).sum():,}",
        ),
        ('Total extracted sentences', f'{total_sentences:,}'),
        ('Sentences with NoRef', f'{num_noref:,}'),
        ('Sentences with ≥1 reference label', f'{num_ref:,}'),
        ('Share of referential sentences', pct(num_ref, total_sentences)),
        ('Sentences with multiple labels', f'{num_multi:,} ({percent_multi}%)'),
        ('Average labels per referential sentence', f'{avg_labels_per_ref:.2f}'),
        ('Traces', f'{len(traces_df):,}'),
        ('Annotations', f'{len(annotations_df):,}'),
        ('Annotation-sentence links', f'{len(sentence_annotation_df):,}'),
    ],
)
display(overview)
category_totals = Counter()
for _, row in sentences_df[sentences_df['has_reference']].iterrows():
    for i, label in enumerate(row['category_vector']):
        if label:
            category_totals[i] += 1
category_distribution_df = pd.DataFrame(
    [
        {
            'category': id2label[i],
            'sentence_count': category_totals[i],
            'percent_of_referential_sentences': round(100 * category_totals[i] / num_ref, 2) if num_ref else 0,
        }
         for i in range(NUM_CATEGORIES)
    ],
)
display(category_distribution_df)
if not multi_label_sentences_df.empty:
    category_combination_distribution = multi_label_sentences_df.assign(
        category_combination=lambda df: df['categories'].apply(lambda cats: ' + '.join(cats)),
    )['category_combination'].value_counts().rename_axis(
        'category_combination',
    ).reset_index(
        name='sentence_count',
    )
    display(category_combination_distribution)
else:
    print('No multi-label sentences found.')
if not annotations_df.empty:
    annotation_category_distribution = annotations_df['category'].fillna('[missing]').value_counts().rename_axis('category').reset_index(
        name='annotation_count',
    )
    display(annotation_category_distribution)
else:
    print('No annotations found.')
if not sentence_annotation_df.empty:
    target_rows = []
    for _, row in sentence_annotation_df.iterrows():
        for target in as_list(row['target_novel']):
            if target:
                target_rows.append({'target_novel': target, 'category': row['category']})
    target_df = pd.DataFrame(target_rows)
    if not target_df.empty:
        target_distribution = target_df['target_novel'].value_counts().rename_axis('target_novel').reset_index(
            name='count',
        )
        display(target_distribution)
    else:
        print('No target-novel data found.')
else:
    print('No sentence-annotation links found.')
sentences_with_annotation_overlap_but_no_valid_category = sentences_df[(sentences_df['overlapping_annotation_count'] > 0) & ~sentences_df['has_reference']]
sentences_with_invalid_categories = sentences_df[sentences_df['invalid_categories'].apply(lambda x: len(x) > 0)]
sanity_checks = make_overview_table(
    [
        (
            'Sentences with annotation overlap but no valid category',
            f'{len(sentences_with_annotation_overlap_but_no_valid_category):,}',
        ),
        (
            'Sentences with invalid/unmapped categories',
            f'{len(sentences_with_invalid_categories):,}',
        ),
        (
            'Negative-example letters with annotations',
            f"{(letters_df['is_negative_example'] & (letters_df['annotation_count'] > 0)).sum():,}",
        ),
        (
            'Non-negative letters without annotations',
            f"{(~letters_df['is_negative_example'] & (letters_df['annotation_count'] == 0)).sum():,}",
        ),
    ],
)
display(sanity_checks)


## Years

In [ ]:
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from shared.metadata import extract_year
from annotations.tables import has_valid_reference_annotation

records = []
for letter in letters:
    reviewed = getattr(letter, 'reviewed', False)
    has_reference = has_valid_reference_annotation(letter)
    if not reviewed:
        annotation_status = 'Not annotated'
    elif has_reference:
        annotation_status = 'Annotated: Ref'
    else:
        annotation_status = 'Annotated: NoRef'
    records.append(
        {
            'id': str(letter.id),
            'year_raw': getattr(letter, 'year', None),
            'year': extract_year(getattr(letter, 'year', None)),
            'recipient': getattr(letter, 'recipient', None),
            'norm_recipient': getattr(letter, 'norm_recipient', None),
            'reviewed': reviewed,
            'is_negative_example': getattr(letter, 'is_negative_example', False),
            'has_reference': has_reference,
            'annotation_status': annotation_status,
        },
    )
df = pd.DataFrame(records)
df_valid = df[df['year'].between(1820, 1870, inclusive='both')].copy()
status_order = ['Annotated: Ref', 'Annotated: NoRef', 'Not annotated']
year_status_counts = df_valid.groupby(['year', 'annotation_status']).size().unstack(fill_value=0).reindex(
    index=range(1820, 1871),
    fill_value=0,
).reindex(
    columns=status_order,
    fill_value=0,
).reset_index()
year_status_counts['total'] = year_status_counts[status_order].sum(axis=1)
print(f'Total letters in object list: {len(df):,}')
print(f'Letters with usable years between 1820 and 1870: {len(df_valid):,}')
print(f'Letters without usable year in this range: {len(df) - len(df_valid):,}')
print('\nAnnotation status totals:')
print(df_valid['annotation_status'].value_counts().reindex(status_order, fill_value=0).to_string())
print('\nYearly letter counts by annotation status:')
print(year_status_counts[['year', 'total'] + status_order].to_string(index=False))
events = {
    1836: 'Early literary career',
    1842: 'First American tour',
    1844: 'Italy and Switzerland',
    1846: 'Daily News',
    1850: 'Household Words',
    1858: 'Public readings',
    1859: 'All the Year Round',
    1867: 'Second American tour',
    1870: 'Death',
}
label_fs = 16
tick_fs = 14
legend_fs = 14
event_fs = 16
title_fs = 18
fig, ax = plt.subplots(figsize=(14, 8))
bottom = pd.Series(0, index=year_status_counts.index)
colors = {'Annotated: Ref': '0.18', 'Annotated: NoRef': '0.55', 'Not annotated': '0.82'}
labels = {
    'Annotated: Ref': 'Annotated: Ref',
    'Annotated: NoRef': 'Annotated: NoRef',
    'Not annotated': 'Not annotated',
}
for status in status_order:
    ax.bar(
        year_status_counts['year'],
        year_status_counts[status],
        bottom=bottom,
        width=0.85,
        color=colors[status],
        label=labels[status],
    )
    bottom += year_status_counts[status]
ymax = year_status_counts['total'].max()
ax.set_ylim(0, ymax * 1.52)
for year, label in events.items():
    ax.axvline(year, color='0.76', linewidth=1, zorder=0)
    x_text = year + 0.15
    y_text = ymax * 1.45
    ax.text(
        x_text,
        y_text,
        label,
        rotation=90,
        va='top',
        ha='left',
        fontsize=event_fs,
        color='0.20',
        linespacing=0.9,
    )
ax.set_xlabel('Year', fontsize=label_fs)
ax.set_ylabel('Number of letters', fontsize=label_fs)
ax.set_xlim(1819.5, 1870.5)
ax.set_xticks(range(1820, 1871, 5))
ax.tick_params(axis='both', labelsize=tick_fs)
ax.legend(frameon=False, loc='upper left', fontsize=legend_fs)
ax.grid(axis='y', color='0.86', linewidth=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
output_dir = Path(str(paths.FIGURES_DIR))
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'annotation_status_distribution_by_year.png', dpi=300, bbox_inches='tight')
plt.show()


## Recipients

In [ ]:
import re
import textwrap
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

TOP_N = 20
output_dir = Path(str(paths.FIGURES_DIR))
output_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_PLOT = output_dir / 'annotation_status_distribution_by_reference_recipient.png'
corpus_letters = letters
from shared.metadata import get_letter_id
from shared.metadata import get_norm_recipient
from shared.metadata import prettify_recipient
from shared.metadata import wrap_label
from annotations.tables import has_valid_reference_annotation


def get_annotation_status(letter):
    is_annotated = getattr(letter, 'reviewed', False)
    has_reference = has_valid_reference_annotation(letter)
    if not is_annotated:
        return 'Not annotated'
    if has_reference:
        return 'Annotated: Ref'
    return 'Annotated: NoRef'


def get_rank_change(global_rank, reference_rank):
    if reference_rank < global_rank:
        return 'up'
    if reference_rank > global_rank:
        return 'down'
    return 'same'
status_order = ['Annotated: Ref', 'Annotated: NoRef', 'Not annotated']
status_labels = {
    'Annotated: Ref': 'Annotated: reference',
    'Annotated: NoRef': 'Annotated: no reference',
    'Not annotated': 'Not annotated',
}
status_colors = {'Annotated: Ref': '0.18', 'Annotated: NoRef': '0.55', 'Not annotated': '0.82'}
records = []
for letter in corpus_letters:
    records.append(
        {
            'id': get_letter_id(letter),
            'norm_recipient': get_norm_recipient(letter),
            'annotated': getattr(letter, 'reviewed', False),
            'has_reference': has_valid_reference_annotation(letter),
            'annotation_status': get_annotation_status(letter),
        },
    )
df = pd.DataFrame(records)
df_rec = df.dropna(subset=['norm_recipient']).copy()
global_counts = df_rec.groupby('norm_recipient').size().rename('global_total').reset_index().sort_values(
    ['global_total', 'norm_recipient'],
    ascending=[False, True],
).reset_index(
    drop=True,
)
global_counts['global_rank'] = global_counts.index + 1
reference_counts = df_rec[df_rec['annotation_status'] == 'Annotated: Ref'].groupby('norm_recipient').size().rename(
    'reference_total',
).reset_index().merge(
    global_counts[['norm_recipient', 'global_rank']],
    on='norm_recipient',
    how='left',
).sort_values(
    ['reference_total', 'global_rank', 'norm_recipient'],
    ascending=[False, True, True],
).reset_index(
    drop=True,
)
reference_counts['reference_rank'] = reference_counts.index + 1
reference_counts = reference_counts[['norm_recipient', 'reference_total', 'reference_rank']]
status_counts = df_rec.groupby(['norm_recipient', 'annotation_status']).size().unstack(fill_value=0).reindex(
    columns=status_order,
    fill_value=0,
).reset_index()
comparison = status_counts.merge(global_counts, on='norm_recipient', how='left').merge(
    reference_counts,
    on='norm_recipient',
    how='left',
)
for status in status_order:
    comparison[status] = comparison[status].fillna(0).astype(int)
comparison['reference_total'] = comparison['reference_total'].fillna(0).astype(int)
comparison['global_total'] = comparison[status_order].sum(axis=1)
plot_df = comparison.dropna(subset=['reference_rank']).copy()
plot_df['reference_rank'] = plot_df['reference_rank'].astype(int)
plot_df = plot_df[plot_df['reference_rank'] <= TOP_N].sort_values('reference_rank').reset_index(drop=True)
plot_df['y_pos'] = range(len(plot_df))
plot_df['recipient_display'] = plot_df['norm_recipient'].apply(prettify_recipient)
plot_df['recipient_display_wrapped'] = plot_df['recipient_display'].apply(wrap_label)
plot_df['rank_change'] = plot_df.apply(
    lambda row: get_rank_change(int(row['global_rank']), int(row['reference_rank'])),
    axis=1,
)
print(f'Top {TOP_N} recipients by annotated reference letters:')
print(
    plot_df[[
        'recipient_display',
        'global_rank',
        'reference_rank',
        'global_total',
        'reference_total',
        'Annotated: Ref',
        'Annotated: NoRef',
        'Not annotated',
        'rank_change',
    ]].to_string(
        index=False,
    ),
)
label_fs = 16
tick_fs = 14
legend_fs = 13
meta_fs = 13
header_fs = 12
fig, ax = plt.subplots(figsize=(15.2, 9.5))
left = pd.Series(0, index=plot_df.index)
for status in status_order:
    ax.barh(
        plot_df['y_pos'],
        plot_df[status],
        left=left,
        height=0.78,
        color=status_colors[status],
        label=status_labels[status],
    )
    left += plot_df[status]
xmax = plot_df['global_total'].max()
x_global_rank = xmax * 1.03
x_reference_rank = xmax * 1.2
ax.set_xlim(0, xmax * 1.4)
for _, row in plot_df.iterrows():
    y = row['y_pos']
    global_rank_label = f"#{int(row['global_rank'])}"
    if row['rank_change'] == 'up':
        reference_rank_label = f"▲ #{int(row['reference_rank'])}"
        reference_rank_color = '#2E8B57'
        reference_rank_weight = 'bold'
    elif row['rank_change'] == 'down':
        reference_rank_label = f"▼ #{int(row['reference_rank'])}"
        reference_rank_color = '#B22222'
        reference_rank_weight = 'bold'
    else:
        reference_rank_label = f"• #{int(row['reference_rank'])}"
        reference_rank_color = '0.35'
        reference_rank_weight = 'normal'
    ax.text(
        x_global_rank,
        y,
        global_rank_label,
        va='center',
        ha='left',
        fontsize=meta_fs,
        color='0.20',
    )
    ax.text(
        x_reference_rank,
        y,
        reference_rank_label,
        va='center',
        ha='left',
        fontsize=meta_fs,
        color=reference_rank_color,
        fontweight=reference_rank_weight,
    )
ax.text(
    x_global_rank,
    -1.22,
    'Global\nrank',
    va='bottom',
    ha='left',
    fontsize=header_fs,
    color='0.20',
    linespacing=0.9,
)
ax.text(
    x_reference_rank,
    -1.22,
    'Reference\nrank',
    va='bottom',
    ha='left',
    fontsize=header_fs,
    color='0.20',
    linespacing=0.9,
)
ax.set_yticks(plot_df['y_pos'])
ax.set_yticklabels(plot_df['recipient_display_wrapped'], fontsize=tick_fs)
ax.invert_yaxis()
ax.set_xlabel('Number of letters', fontsize=label_fs)
ax.set_ylabel('Recipient', fontsize=label_fs)
ax.tick_params(axis='x', labelsize=tick_fs)
ax.grid(axis='x', color='0.86', linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.legend(
    handles=[
        Patch(facecolor=status_colors['Annotated: Ref'], label='Annotated: reference'),
        Patch(facecolor=status_colors['Annotated: NoRef'], label='Annotated: no reference'),
        Patch(facecolor=status_colors['Not annotated'], label='Not annotated'),
    ],
    frameon=False,
    ncol=3,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.985),
    fontsize=legend_fs,
    columnspacing=1.8,
    handlelength=2.6,
)
fig.subplots_adjust(top=0.86, right=0.94, left=0.23)
fig.savefig(OUTPUT_PLOT, dpi=300, bbox_inches='tight')
plt.show()
print(f'\nSaved plot to: {OUTPUT_PLOT}')


## Category combinations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from upsetplot import UpSet

MIN_COMBINATION_COUNT = 3
FONT_SIZE_INCREASE = 3
CATEGORY_ORDER_TOP_TO_BOTTOM = [
    'Inspirations and Influences',
    'Writing Process and Creative Insight',
    'Elements of Fictional Craft',
    'Editorial and Publishing Processes',
    'Quality and Evaluation',
    'Financial and Legal Aspects',
    'Gratitude and Acknowledgement',
    'Marketing and Promotion',
    'Impact and Reception',
]
CATEGORY_PLOT_LABELS = {
    'Editorial and Publishing Processes': 'Editorial/Publishing',
    'Elements of Fictional Craft': 'Fictional Craft',
    'Financial and Legal Aspects': 'Financial/Legal',
    'Gratitude and Acknowledgement': 'Gratitude',
    'Impact and Reception': 'Impact/Reception',
    'Inspirations and Influences': 'Inspirations',
    'Marketing and Promotion': 'Marketing',
    'Quality and Evaluation': 'Quality/Evaluation',
    'Writing Process and Creative Insight': 'Writing Process',
}
category_by_index = [label for label, _ in sorted(CATEGORY_INDEX.items(), key=lambda item: item[1])]
all_plot_categories = [CATEGORY_PLOT_LABELS[label] for label in CATEGORY_ORDER_TOP_TO_BOTTOM]
ref_sentences = sentences_df[sentences_df['has_reference']].copy()
if ref_sentences.empty:
    raise ValueError('No referential sentences found in sentences_df.')


def vector_to_original_categories(category_vector):
    return [category_by_index[i] for i, value in enumerate(category_vector) if value == 1]


def vector_to_plot_categories(category_vector):
    return [
        CATEGORY_PLOT_LABELS[category_by_index[i]]
         for i, value in enumerate(category_vector) if value == 1
    ]
ref_sentences['categories_original'] = ref_sentences['category_vector'].apply(vector_to_original_categories)
ref_sentences['categories_plot'] = ref_sentences['category_vector'].apply(vector_to_plot_categories)
ref_sentences['category_degree'] = ref_sentences['categories_plot'].apply(len)
ref_sentences['combination_key'] = ref_sentences['categories_plot'].apply(tuple)
isolated_counts_unsorted = ref_sentences.loc[ref_sentences['category_degree'] == 1, 'categories_plot'].apply(
    lambda cats: cats[0],
).value_counts().reindex(
    all_plot_categories,
    fill_value=0,
)
visible_category_order_plot = isolated_counts_unsorted.sort_values(ascending=False, kind='mergesort').index.tolist()
category_order_plot = list(reversed(visible_category_order_plot))
isolated_counts = isolated_counts_unsorted.reindex(category_order_plot)
combination_counts = ref_sentences.loc[ref_sentences['category_degree'] >= 2, 'combination_key'].value_counts()
displayed_combination_keys = combination_counts[combination_counts >= MIN_COMBINATION_COUNT].index
plot_sentences = ref_sentences[(ref_sentences['category_degree'] >= 2) & ref_sentences['combination_key'].isin(displayed_combination_keys)].copy()
if plot_sentences.empty:
    raise ValueError(f'No category combinations found with count >= {MIN_COMBINATION_COUNT}.')
membership_matrix = pd.DataFrame(
    [
        {plot_label: plot_label in categories for plot_label in category_order_plot}
         for categories in plot_sentences['categories_plot']
    ],
)
upset_input = membership_matrix.copy()
upset_input['sentence_row_id'] = np.arange(len(upset_input))
upset_input = upset_input.set_index(category_order_plot)
print('\nIsolated category counts, sorted by visible plot order:')
print(
    isolated_counts_unsorted.reindex(visible_category_order_plot).rename_axis('category').reset_index(
        name='isolated_count',
    ).to_string(
        index=False,
    ),
)
print('\nDisplayed combination counts:')
displayed_combination_counts = combination_counts[combination_counts >= MIN_COMBINATION_COUNT].sort_values(ascending=False)
displayed_combination_summary = displayed_combination_counts.rename_axis('combination').reset_index(name='count')
displayed_combination_summary['combination'] = displayed_combination_summary['combination'].apply(lambda combo: ' + '.join(combo))
print(displayed_combination_summary.to_string(index=False))
print(
    f'\nDisplayed combinations: {len(displayed_combination_counts)} of {len(combination_counts)} total multi-category combinations.',
)
fig = plt.figure(figsize=(11.69, 8.27))
upset = UpSet(
    upset_input,
    subset_size='count',
    sort_by='cardinality',
    sort_categories_by='input',
    min_degree=2,
    show_counts='%d',
    element_size=40,
    intersection_plot_elements=7,
    totals_plot_elements=6,
)
axes = upset.plot(fig=fig)
ax_totals = axes['totals']
ax_matrix = axes['matrix']
plotted_categories = list(upset.totals.index)
isolated_for_axis = isolated_counts.reindex(plotted_categories, fill_value=0).astype(int)
ax_totals.clear()
y_positions = np.arange(len(plotted_categories))
max_isolated = max(isolated_for_axis.max(), 1)
label_offset = max_isolated * 0.025
ax_totals.barh(y_positions, isolated_for_axis.values, height=0.5, color='black', align='center')
ax_totals.set_yticks(y_positions)
ax_totals.set_yticklabels(plotted_categories, fontsize=10)
ax_totals.tick_params(axis='y', pad=8)
for y, value in zip(y_positions, isolated_for_axis.values):
    ax_totals.text(
        value + label_offset,
        y,
        str(int(value)),
        va='center',
        ha='right',
        fontsize=10,
        clip_on=False,
    )
ax_totals.set_xlabel('Isolated category count', labelpad=12)
ax_totals.set_xlim(max_isolated * 1.3, 0)
ax_totals.set_ylim(ax_matrix.get_ylim())
ax_totals.spines['top'].set_visible(False)
ax_totals.spines['right'].set_visible(False)
axes['intersections'].set_ylabel('Combination count')
axes['matrix'].set_xlabel('Category combinations')
for text in fig.findobj(match=plt.Text):
    if text.get_text():
        text.set_fontsize(text.get_fontsize() + FONT_SIZE_INCREASE)
fig.subplots_adjust(top=0.97, bottom=0.08)
plt.savefig(
    paths.FIGURES_DIR / 'reference_category_combinations.png',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.2,
)
plt.show()


## Markers

In [ ]:
from pathlib import Path
from collections import Counter
import re
import pandas as pd
import matplotlib.pyplot as plt

MARKER_ATTRIBUTE_CANDIDATES = [
    'markers',
    'marker',
    'reference_markers',
    'reference_marker',
    'intertextual_markers',
    'intertextual_marker',
    'marker_types',
    'marker_type',
]
output_dir = Path(str(paths.FIGURES_DIR))
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 'trace_marker_distribution.png'
label_fs = 16
tick_fs = 14
value_fs = 14
title_fs = 18
from annotations.tables import as_list


def clean_marker_value(value):
    if value is None:
        return None
    value = str(value).strip()
    if not value:
        return None
    return value


def format_marker_label(value):
    value = str(value).strip()
    value = re.sub('[_\\-]+', ' ', value)
    value = re.sub('\\s+', ' ', value)
    acronym_map = {'tei': 'TEI', 'xml': 'XML', 'ocr': 'OCR'}
    words = []
    for word in value.split():
        lower = word.lower()
        if lower in acronym_map:
            words.append(acronym_map[lower])
        else:
            words.append(word.capitalize())
    return ' '.join(words)


def find_marker_attribute(reviewed_letters, candidate_attributes):
    found = Counter()
    example_attrs = []
    for letter in reviewed_letters:
        traces = getattr(letter, 'intertextual_traces', []) or []
        for trace in traces:
            trace_attrs = vars(trace).keys() if hasattr(trace, '__dict__') else dir(trace)
            if not example_attrs:
                example_attrs = sorted((attr for attr in trace_attrs if not attr.startswith('_')))
            for attr in candidate_attributes:
                if hasattr(trace, attr):
                    value = getattr(trace, attr)
                    if value is not None:
                        found[attr] += 1
    if found:
        return found.most_common(1)[0][0]
    print('\nNo marker attribute was found using the current candidate list.')
    print('\nAvailable attributes on the first trace object:')
    for attr in example_attrs:
        print(f'- {attr}')
    raise AttributeError(
        'Could not find a marker attribute on intertextual traces. Check the printed trace attributes and add the correct name to MARKER_ATTRIBUTE_CANDIDATES.',
    )
marker_attribute = find_marker_attribute(reviewed_letters, MARKER_ATTRIBUTE_CANDIDATES)
print(f'\nUsing trace-level marker attribute: {marker_attribute}')
marker_rows = []
trace_rows = []
for letter in reviewed_letters:
    traces = getattr(letter, 'intertextual_traces', []) or []
    for trace_index, trace in enumerate(traces):
        trace_id = f'{letter.id}__t{trace_index}'
        raw_markers = getattr(trace, marker_attribute, None)
        markers = [clean_marker_value(marker) for marker in as_list(raw_markers)]
        markers = [marker for marker in markers if marker is not None]
        trace_rows.append(
            {
                'trace_id': trace_id,
                'letter_id': letter.id,
                'trace_index': trace_index,
                'marker_count': len(markers),
                'markers': markers,
            },
        )
        for marker in markers:
            marker_rows.append(
                {
                    'trace_id': trace_id,
                    'letter_id': letter.id,
                    'trace_index': trace_index,
                    'marker_raw': marker,
                    'marker': format_marker_label(marker),
                },
            )
trace_marker_df = pd.DataFrame(trace_rows)
marker_assignment_df = pd.DataFrame(marker_rows)
total_traces = len(trace_marker_df)
traces_with_marker = trace_marker_df['marker_count'].gt(0).sum() if not trace_marker_df.empty else 0
total_marker_assignments = len(marker_assignment_df)
print('\nMarker extraction summary:')
print(f'Intertextual traces: {total_traces:,}')
print(f'Traces with at least one marker: {traces_with_marker:,}')
print(f'Marker assignments: {total_marker_assignments:,}')
if total_traces:
    print(f'Share of traces with at least one marker: {traces_with_marker / total_traces:.2%}')
if marker_assignment_df.empty:
    raise ValueError('No marker assignments found.')
marker_summary = marker_assignment_df.groupby('marker').agg(
    trace_count=('trace_id', 'nunique'),
    marker_assignments=('marker', 'size'),
).reset_index()
marker_summary['share_of_all_traces'] = marker_summary['trace_count'] / total_traces * 100
marker_summary['share_of_traces_with_marker'] = marker_summary['trace_count'] / traces_with_marker * 100
marker_summary = marker_summary.sort_values('trace_count', ascending=False).reset_index(drop=True)
print('\nMarker distribution by intertextual traces:')
print(
    marker_summary.assign(
        share_of_all_traces=lambda df: df['share_of_all_traces'].round(2),
        share_of_traces_with_marker=lambda df: df['share_of_traces_with_marker'].round(2),
    ).to_string(
        index=False,
    ),
)
plot_df = marker_summary.sort_values('trace_count', ascending=True).copy()
fig_height = max(5, 0.55 * len(plot_df))
fig, ax = plt.subplots(figsize=(11, fig_height))
ax.barh(plot_df['marker'], plot_df['trace_count'], color='0.18', height=0.72)
max_traces = plot_df['trace_count'].max()
label_offset = max(max_traces * 0.012, 0.5)
for y_pos, value in enumerate(plot_df['trace_count']):
    ax.text(
        value + label_offset,
        y_pos,
        str(int(value)),
        va='center',
        ha='left',
        fontsize=value_fs,
        color='0.10',
    )
ax.set_xlabel('Intertextual traces', fontsize=label_fs)
ax.set_ylabel('Marker', fontsize=label_fs)
ax.tick_params(axis='both', labelsize=tick_fs)
ax.grid(axis='x', color='0.86', linewidth=0.8)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, max_traces * 1.14)
fig.tight_layout()
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()


## Target works

In [ ]:
import pandas as pd

TOP_N_TARGETS = 20
from annotations.tables import as_list


def clean_target_value(value):
    if value is None:
        return None
    value = str(value).strip()
    if not value:
        return None
    return value
ref_sentences = sentences_df[sentences_df['has_reference']].copy()
if ref_sentences.empty:
    raise ValueError('No referential sentences found in sentences_df.')
target_rows = []
for _, row in ref_sentences.iterrows():
    sentence_uid = f"{row['letter_id']}__s{row['sentence_id']}"
    targets = [clean_target_value(target) for target in as_list(row.get('target_novels', []))]
    targets = sorted({target for target in targets if target is not None})
    for target in targets:
        target_rows.append(
            {
                'sentence_uid': sentence_uid,
                'letter_id': row['letter_id'],
                'sentence_id': row['sentence_id'],
                'target_work': target,
            },
        )
target_sentence_df = pd.DataFrame(target_rows)
total_sentences = len(sentences_df)
total_ref_sentences = len(ref_sentences)
if target_sentence_df.empty:
    raise ValueError("No target works found in sentences_df['target_novels'].")
sentences_with_target = target_sentence_df['sentence_uid'].nunique()
target_assignments = len(target_sentence_df)
print('\nTarget work extraction summary:')
print(f'All extracted sentences: {total_sentences:,}')
print(f'Referential sentences: {total_ref_sentences:,}')
print(f'Referential sentences with at least one target work: {sentences_with_target:,}')
print(f'Sentence-level target-work assignments: {target_assignments:,}')
print(
    f'Share of referential sentences with at least one target work: {sentences_with_target / total_ref_sentences:.2%}',
)
target_summary = target_sentence_df.groupby('target_work').agg(
    sentence_count=('sentence_uid', 'nunique'),
    target_assignments=('target_work', 'size'),
).reset_index()
target_summary['share_of_ref_sentences'] = target_summary['sentence_count'] / total_ref_sentences * 100
target_summary['share_of_target_sentences'] = target_summary['sentence_count'] / sentences_with_target * 100
target_summary = target_summary.sort_values(['sentence_count', 'target_work'], ascending=[False, True]).reset_index(
    drop=True,
)
print('\nFull target work distribution on sentence level:')
print(
    target_summary.assign(
        share_of_ref_sentences=lambda df: df['share_of_ref_sentences'].round(2),
        share_of_target_sentences=lambda df: df['share_of_target_sentences'].round(2),
    ).to_string(
        index=False,
    ),
)
top_targets = target_summary.head(TOP_N_TARGETS).copy()
other_targets = target_summary.iloc[TOP_N_TARGETS:].copy()
if not other_targets.empty:
    other_row = pd.DataFrame(
        [
            {
                'target_work': 'Other target works',
                'sentence_count': other_targets['sentence_count'].sum(),
                'target_assignments': other_targets['target_assignments'].sum(),
                'share_of_ref_sentences': other_targets['sentence_count'].sum() / total_ref_sentences * 100,
                'share_of_target_sentences': other_targets['sentence_count'].sum() / sentences_with_target * 100,
            },
        ],
    )
    compact_target_table = pd.concat([top_targets, other_row], ignore_index=True)
else:
    compact_target_table = top_targets.copy()
print(f'\nCompact target work overview, top {TOP_N_TARGETS} plus Other:')
print(
    compact_target_table.assign(
        share_of_ref_sentences=lambda df: df['share_of_ref_sentences'].round(1),
        share_of_target_sentences=lambda df: df['share_of_target_sentences'].round(1),
    )[['target_work', 'sentence_count', 'share_of_ref_sentences', 'share_of_target_sentences']].to_string(
        index=False,
    ),
)


## Temporal perspective

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

temporal_counts = Counter(
    (str(value).strip().capitalize() for row in all_sentences for value in row.get('temporal_perspectives', []) if str(value).strip()),
)
temporal_df = pd.DataFrame(
    {
        'temporal_perspective': list(temporal_counts.keys()),
        'instances': list(temporal_counts.values()),
    },
)
total_instances = temporal_df['instances'].sum()
temporal_df['share'] = temporal_df['instances'] / total_instances * 100
temporal_df['order'] = temporal_df['temporal_perspective'].map({'Analeptic': 0, 'Current': 1, 'Proleptic': 2})
temporal_df = temporal_df.sort_values('order').reset_index(drop=True)
output_dir = Path(str(paths.FIGURES_DIR))
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 'temporal_perspective_distribution.png'
label_fs = 16
tick_fs = 14
value_fs = 14
print('\nTemporal perspective distribution:')
print(
    temporal_df[['temporal_perspective', 'instances', 'share']].assign(
        share=lambda df: df['share'].round(2),
    ).to_string(
        index=False,
    ),
)
print(f'\nTotal plotted instances: {total_instances:,}')
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.barh(temporal_df['temporal_perspective'], temporal_df['instances'], color='0.18', height=0.62)
max_instances = temporal_df['instances'].max()
label_offset = max_instances * 0.018
for y_pos, row in temporal_df.iterrows():
    ax.text(
        row['instances'] + label_offset,
        y_pos,
        f"{int(row['instances']):,} ({row['share']:.1f}%)",
        va='center',
        ha='left',
        fontsize=value_fs,
        color='0.10',
    )
ax.set_xlabel('Number of Assignments', fontsize=label_fs)
ax.set_ylabel('Temporal perspective', fontsize=label_fs)
ax.tick_params(axis='both', labelsize=tick_fs)
ax.grid(axis='x', color='0.86', linewidth=0.8)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, max_instances * 1.22)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
